In [4]:
import pandas as pd

df = pd.read_csv('../data/processed/ndbc_data_processed.csv')
df.head()

,timestamp,wind_speed_ms,wind_gust_ms,air_temp_c,water_temp_c,dewpoint_c,wind_direction_deg,air_pressure_hpa
0,2014-02-01 01:00:00,5.30,6.35,3.90,4.55,-1.9,281.0,1022.05
1,2014-02-01 02:00:00,5.50,6.35,3.60,4.55,-4.4,283.0,1022.70
2,2014-02-01 03:00:00,4.25,5.15,3.50,4.50,-4.7,269.5,1023.20
3,2014-02-01 04:00:00,4.55,5.55,3.75,4.45,-5.1,279.5,1023.85
4,2014-02-01 05:00:00,4.30,4.85,3.55,4.40,-5.1,262.0,1024.05


## Add sunrise/sunset, moonrise/moonset, sun/moon distance, and moon phase

This adds the requested astronomy columns for each row in `df`.

Before running:
1. Install dependencies in your environment: `pip install astral astropy`
2. Set `OBS_LAT`, `OBS_LON`, and `LOCAL_TZ` for your station/location.

In [5]:
import pandas as pd
import numpy as np

from astral import Observer
from astral.sun import sun
from astral.moon import moonrise, moonset, phase

from astropy.time import Time
from astropy.coordinates import get_sun, get_body, EarthLocation
import astropy.units as u

# ---- Configure for your location ----
OBS_LAT = 40.3233
OBS_LON = -74.0757
LOCAL_TZ = "America/New_York"
TIMESTAMP_FMT = "%Y-%m-%d %H:%M:%S"

# ---- Parse timestamps ----
df = df.copy()
df["timestamp_utc"] = pd.to_datetime(df["timestamp"], utc=True)
df["timestamp_local"] = df["timestamp_utc"].dt.tz_convert(LOCAL_TZ)
df["date_local"] = df["timestamp_local"].dt.date

# Make cell re-runnable without creating suffixed duplicate columns.
astronomy_cols = [
    "sunrise_time",
    "sunset_time",
    "moonrise_time",
    "moonset_time",
    "moon_phase_age_days",
    "moon_phase_name",
    "sun_distance_km",
    "moon_distance_km",
]
df = df.drop(columns=[c for c in astronomy_cols if c in df.columns], errors="ignore")

observer = Observer(latitude=OBS_LAT, longitude=OBS_LON)


def safe_events_for_day(day):
    """Return local sunrise/sunset/moonrise/moonset + moon phase for a day."""
    try:
        s = sun(observer, date=day, tzinfo=LOCAL_TZ)
        sunrise_t = s["sunrise"]
        sunset_t = s["sunset"]
    except Exception:
        sunrise_t = pd.NaT
        sunset_t = pd.NaT

    try:
        moonrise_t = moonrise(observer, date=day, tzinfo=LOCAL_TZ)
    except Exception:
        moonrise_t = pd.NaT

    try:
        moonset_t = moonset(observer, date=day, tzinfo=LOCAL_TZ)
    except Exception:
        moonset_t = pd.NaT

    moon_phase_age_days = float(phase(day))

    return pd.Series(
        {
            "sunrise_time": sunrise_t,
            "sunset_time": sunset_t,
            "moonrise_time": moonrise_t,
            "moonset_time": moonset_t,
            "moon_phase_age_days": moon_phase_age_days,
        }
    )


# Compute local daily events once, then merge back to each row by local date.
daily_events = pd.DataFrame({"date_local": df["date_local"].unique()})
daily_events = daily_events.join(daily_events["date_local"].apply(safe_events_for_day))

df = df.merge(daily_events, on="date_local", how="left")

# Format event-time columns like the source timestamp: YYYY-MM-DD HH:MM:SS
for col in ["sunrise_time", "sunset_time", "moonrise_time", "moonset_time"]:
    df[col] = df[col].apply(
        lambda x: x.tz_localize(None).strftime(TIMESTAMP_FMT) if pd.notna(x) else pd.NA
    )

# ---- Distance to Sun/Moon at each row timestamp ----
location = EarthLocation(lat=OBS_LAT * u.deg, lon=OBS_LON * u.deg, height=0 * u.m)
obs_times = Time(df["timestamp_utc"].to_numpy(dtype="datetime64[ns]"), format="datetime64", scale="utc")

sun_coord = get_sun(obs_times)
moon_coord = get_body("moon", obs_times, location=location)

df["sun_distance_km"] = sun_coord.distance.to(u.km).value
df["moon_distance_km"] = moon_coord.distance.to(u.km).value

# Optional: compact moon phase label for modeling/EDA
phase_bins = [-0.1, 1.8, 5.5, 9.2, 12.9, 16.6, 20.3, 24.0, 28.1]
phase_labels = [
    "new_moon",
    "waxing_crescent",
    "first_quarter",
    "waxing_gibbous",
    "full_moon",
    "waning_gibbous",
    "last_quarter",
    "waning_crescent",
]
df["moon_phase_name"] = pd.cut(
    df["moon_phase_age_days"], bins=phase_bins, labels=phase_labels
).astype("string")

cols_added = [
    "sunrise_time",
    "sunset_time",
    "moonrise_time",
    "moonset_time",
    "sun_distance_km",
    "moon_distance_km",
    "moon_phase_age_days",
    "moon_phase_name",
]

print("Added columns:", cols_added)
df[cols_added + ["timestamp_local", "timestamp_utc"]].head()

Added columns: ['sunrise_time', 'sunset_time', 'moonrise_time', 'moonset_time', 'sun_distance_km', 'moon_distance_km', 'moon_phase_age_days', 'moon_phase_name']


,sunrise_time,sunset_time,moonrise_time,moonset_time,sun_distance_km,moon_distance_km,moon_phase_age_days,moon_phase_name,timestamp_local,timestamp_utc
0,2014-01-31 07:06:39,2014-01-31 17:13:21,2014-01-31 07:14:00,2014-01-31 18:35:00,1.474073e+08,361404.808532,0.577889,new_moon,2014-01-31 20:00:00-05:00,2014-02-01 01:00:00+00:00
1,2014-01-31 07:06:39,2014-01-31 17:13:21,2014-01-31 07:14:00,2014-01-31 18:35:00,1.474081e+08,362653.467797,0.577889,new_moon,2014-01-31 21:00:00-05:00,2014-02-01 02:00:00+00:00
2,2014-01-31 07:06:39,2014-01-31 17:13:21,2014-01-31 07:14:00,2014-01-31 18:35:00,1.474090e+08,363762.673757,0.577889,new_moon,2014-01-31 22:00:00-05:00,2014-02-01 03:00:00+00:00
3,2014-01-31 07:06:39,2014-01-31 17:13:21,2014-01-31 07:14:00,2014-01-31 18:35:00,1.474098e+08,364671.158659,0.577889,new_moon,2014-01-31 23:00:00-05:00,2014-02-01 04:00:00+00:00
4,2014-02-01 07:05:43,2014-02-01 17:14:34,2014-02-01 07:54:00,<NA>,1.474107e+08,365330.514787,1.666778,new_moon,2014-02-01 00:00:00-05:00,2014-02-01 05:00:00+00:00


In [6]:
df.head()

,timestamp,wind_speed_ms,wind_gust_ms,air_temp_c,water_temp_c,dewpoint_c,wind_direction_deg,air_pressure_hpa,timestamp_utc,timestamp_local,date_local,sunrise_time,sunset_time,moonrise_time,moonset_time,moon_phase_age_days,sun_distance_km,moon_distance_km,moon_phase_name
0,2014-02-01 01:00:00,5.30,6.35,3.90,4.55,-1.9,281.0,1022.05,2014-02-01 01:00:00+00:00,2014-01-31 20:00:00-05:00,2014-01-31,2014-01-31 07:06:39,2014-01-31 17:13:21,2014-01-31 07:14:00,2014-01-31 18:35:00,0.577889,1.474073e+08,361404.808532,new_moon
1,2014-02-01 02:00:00,5.50,6.35,3.60,4.55,-4.4,283.0,1022.70,2014-02-01 02:00:00+00:00,2014-01-31 21:00:00-05:00,2014-01-31,2014-01-31 07:06:39,2014-01-31 17:13:21,2014-01-31 07:14:00,2014-01-31 18:35:00,0.577889,1.474081e+08,362653.467797,new_moon
2,2014-02-01 03:00:00,4.25,5.15,3.50,4.50,-4.7,269.5,1023.20,2014-02-01 03:00:00+00:00,2014-01-31 22:00:00-05:00,2014-01-31,2014-01-31 07:06:39,2014-01-31 17:13:21,2014-01-31 07:14:00,2014-01-31 18:35:00,0.577889,1.474090e+08,363762.673757,new_moon
3,2014-02-01 04:00:00,4.55,5.55,3.75,4.45,-5.1,279.5,1023.85,2014-02-01 04:00:00+00:00,2014-01-31 23:00:00-05:00,2014-01-31,2014-01-31 07:06:39,2014-01-31 17:13:21,2014-01-31 07:14:00,2014-01-31 18:35:00,0.577889,1.474098e+08,364671.158659,new_moon
4,2014-02-01 05:00:00,4.30,4.85,3.55,4.40,-5.1,262.0,1024.05,2014-02-01 05:00:00+00:00,2014-02-01 00:00:00-05:00,2014-02-01,2014-02-01 07:05:43,2014-02-01 17:14:34,2014-02-01 07:54:00,<NA>,1.666778,1.474107e+08,365330.514787,new_moon


In [7]:
df.columns.tolist()

['timestamp',
 'wind_speed_ms',
 'wind_gust_ms',
 'air_temp_c',
 'water_temp_c',
 'dewpoint_c',
 'wind_direction_deg',
 'air_pressure_hpa',
 'timestamp_utc',
 'timestamp_local',
 'date_local',
 'sunrise_time',
 'sunset_time',
 'moonrise_time',
 'moonset_time',
 'moon_phase_age_days',
 'sun_distance_km',
 'moon_distance_km',
 'moon_phase_name']

In [8]:
df = df[['timestamp', 'wind_speed_ms', 'wind_gust_ms', 'air_temp_c', 'water_temp_c', 'dewpoint_c',  'wind_direction_deg', 'air_pressure_hpa', 'sunrise_time', 'sunset_time',  'moonrise_time',  'moonset_time',  'moon_phase_age_days',  'sun_distance_km', 'moon_distance_km']]
df.head()

,timestamp,wind_speed_ms,wind_gust_ms,air_temp_c,water_temp_c,dewpoint_c,wind_direction_deg,air_pressure_hpa,sunrise_time,sunset_time,moonrise_time,moonset_time,moon_phase_age_days,sun_distance_km,moon_distance_km
0,2014-02-01 01:00:00,5.30,6.35,3.90,4.55,-1.9,281.0,1022.05,2014-01-31 07:06:39,2014-01-31 17:13:21,2014-01-31 07:14:00,2014-01-31 18:35:00,0.577889,1.474073e+08,361404.808532
1,2014-02-01 02:00:00,5.50,6.35,3.60,4.55,-4.4,283.0,1022.70,2014-01-31 07:06:39,2014-01-31 17:13:21,2014-01-31 07:14:00,2014-01-31 18:35:00,0.577889,1.474081e+08,362653.467797
2,2014-02-01 03:00:00,4.25,5.15,3.50,4.50,-4.7,269.5,1023.20,2014-01-31 07:06:39,2014-01-31 17:13:21,2014-01-31 07:14:00,2014-01-31 18:35:00,0.577889,1.474090e+08,363762.673757
3,2014-02-01 04:00:00,4.55,5.55,3.75,4.45,-5.1,279.5,1023.85,2014-01-31 07:06:39,2014-01-31 17:13:21,2014-01-31 07:14:00,2014-01-31 18:35:00,0.577889,1.474098e+08,364671.158659
4,2014-02-01 05:00:00,4.30,4.85,3.55,4.40,-5.1,262.0,1024.05,2014-02-01 07:05:43,2014-02-01 17:14:34,2014-02-01 07:54:00,<NA>,1.666778,1.474107e+08,365330.514787
